# Hugging Face Fundamentals — Lesson 3: Tokenizer

> Learning material for **Hugging Face Fundamentals**. Companion to the lesson script `03_Tokenizer.py` (same content, runnable without Jupyter).

**Task ID:** HF-003  |  **Folder:** `03_Tokenizer`


## Why do we need numbers?

Neural networks do math with numbers — they cannot see letters. The **tokenizer** is the bridge: it splits text into pieces (tokens), looks each piece up in a vocabulary, and returns an id per piece.

> `"Hugging Face is awesome!"` → tokens → `[17662, 2227, 2003, 12476, 999]`

## Words are split into subwords

BERT uses **WordPiece**: rare words are cut into smaller known pieces:

> `unhappiness` → `un` + `##ha` + `##pp` + `##iness`

The `##` means *this piece continues a word*. That way the model knows `token` and `##izer` belong together — and it can represent words it never saw in training.

**Step 1 — load the tokenizer** (a tokenizer is small: a vocabulary table + rules, no big neural network):


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("vocab size:", tokenizer.vocab_size)


**Step 2 — text → tokens → ids → back to text:**

In [ ]:
text = "Hugging Face is awesome!"
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)
print("text   :", text)
print("tokens :", tokens)
print("ids    :", ids)
print("decoded:", tokenizer.decode(ids))


## Special tokens: the model's punctuation

BERT reserves a few ids for structure — the model *reads* them like punctuation:

| Token | Id | Meaning |
|-------|----|---------|
| `[CLS]` | 101 | start / class token |
| `[SEP]` | 102 | separator between sentences |
| `[PAD]` | 0 | padding filler |
| `[UNK]` | 100 | unknown (not in the vocab) |

Test what happens with characters BERT never saw:


In [ ]:
print(tokenizer.tokenize("unhappiness"))      # subwords
print(tokenizer.tokenize("tokenizer"))         # subwords
print(tokenizer.tokenize("\U0001f600"))         # emoji -> [UNK]


> ▸ The emoji has no entry in the 30,522-token vocabulary, so it becomes `[UNK]` — information is lost. Good tokenizers have bigger vocabularies to avoid this.

## Batching: many texts at once

One call, a list of texts, and padding so every row has equal length:


In [ ]:
enc = tokenizer(["short text", "a much longer text that needs padding"],
                 padding=True, truncation=True)
for t, ids in zip(
    ["short text", "a much longer text that needs padding"], enc["input_ids"]
):
    print(f"{t!r:<44} -> {ids}")


`padding=True` fills short rows with `[PAD]` (id 0); `truncation=True` cuts overlong texts. Models need rectangular input — batches are matrices.

## Try it yourself

1. Tokenize your own name — how many subwords?
2. Tokenize a sentence in your native language (if not English).
3. Find a word that becomes one single token, and one that needs 4+.

## Common pitfalls

- **A token is not a word** — `##` pieces exist; `len(tokens) != len(words)`.
- **`decode()` adds spaces** — by design, it is not a perfect inverse of `tokenize`.
- **`[UNK]` loses data** — prefer models with larger vocabularies for your language.

## Summary

- Tokenizer = vocabulary + splitting rules; text ↔ ids both directions.
- WordPiece cuts rare words into `##`-pieces.
- Special tokens (`[CLS]`, `[SEP]`, `[PAD]`, `[UNK]`) structure the input.

**Next lesson:** HF-004 — AutoTokenizer.  |  Extra reading: `../resources/reference_links.md`
